# SeizureGuard Portfolio — Streamlit depuis Google Colab

Ce notebook lance le site web Streamlit du projet **IoT / IoMT / Edge AI — Groupe 2** sans installation locale.

Il permet de :

1. cloner le dépôt GitHub ;
2. installer les dépendances légères du site ;
3. lancer `streamlit_app.py` ;
4. obtenir une URL publique temporaire via Cloudflare Tunnel ;
5. préparer, si nécessaire, la reproduction complète des résultats après ajout des données SeizeIT2.

> Le mode site fonctionne sans télécharger le dataset brut. Le mode reproduction complète exige les fichiers SeizeIT2 dans `data/`.

In [ ]:
# Paramètres du projet
REPO_URL = "https://github.com/akiroussama/iot-edge-ai-seizure-detection.git"
REPO_DIR = "iot-edge-ai-seizure-detection"
BRANCH = "main"

## 1. Cloner le dépôt

In [ ]:
import os
from pathlib import Path

if Path(REPO_DIR).exists():
    print(f"Suppression de l'ancienne copie : {REPO_DIR}")
    !rm -rf {REPO_DIR}

!git clone --depth 1 -b {BRANCH} {REPO_URL}
%cd {REPO_DIR}

print("\nFichiers principaux :")
!ls -la | sed -n '1,40p'

## 2. Installer les dépendances du site Streamlit

Le fichier `requirements.txt` du repo est volontairement léger pour faciliter le déploiement Streamlit Cloud.

In [ ]:
!python --version
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

print("Dépendances Streamlit installées.")

## 3. Vérifier la présence des livrables et de la trace IA

In [ ]:
from pathlib import Path

expected = [
    "README.md",
    "streamlit_app.py",
    "requirements.txt",
    "presentation/trace_ia.md",
    "docs/REPRODUCTION.md",
    "results/multirun_loso.csv",
    "results/mlp_loso.csv",
    "results/esp32_cost_estimate.json",
]

for item in expected:
    p = Path(item)
    print(f"{item:<38} -> {'OK' if p.exists() else 'absent'}")

## 4. Lancer Streamlit en arrière-plan

In [ ]:
# Arrêter une ancienne session Streamlit si elle existe
!pkill -f "streamlit run" || true

# Lancer le site
!nohup streamlit run streamlit_app.py --server.port 8501 --server.address 0.0.0.0 > /tmp/streamlit.log 2>&1 &
!sleep 4

print("Dernières lignes du log Streamlit :")
!tail -n 30 /tmp/streamlit.log

## 5. Créer une URL publique temporaire

La cellule suivante télécharge `cloudflared`, lance un tunnel vers le port 8501, puis affiche l'URL `trycloudflare.com` à ouvrir dans le navigateur.

In [ ]:
import re
import time
from pathlib import Path

cloudflared = Path("/tmp/cloudflared")
if not cloudflared.exists():
    !wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /tmp/cloudflared

!pkill -f "cloudflared tunnel" || true
!nohup /tmp/cloudflared tunnel --url http://127.0.0.1:8501 --no-autoupdate > /tmp/cloudflared.log 2>&1 &
!sleep 10

log = Path("/tmp/cloudflared.log").read_text(errors="ignore")
urls = re.findall(r"https://[-a-zA-Z0-9.]*trycloudflare\.com", log)
if urls:
    print("URL Streamlit temporaire :")
    print(urls[0])
else:
    print("URL non trouvée pour l'instant. Log Cloudflare :")
    print(log[-2000:])
    print("Relancez cette cellule si nécessaire.")

## 6. Optionnel — tester le site localement dans Colab

Cette cellule vérifie que Streamlit répond bien sur le port 8501.

In [ ]:
import urllib.request

try:
    with urllib.request.urlopen("http://127.0.0.1:8501", timeout=5) as response:
        print("Statut HTTP local :", response.status)
        print("Streamlit répond correctement.")
except Exception as exc:
    print("Streamlit ne répond pas encore :", exc)
    print("Regardez le log :")
    !tail -n 80 /tmp/streamlit.log

## 7. Optionnel — reproduction complète des résultats

À exécuter seulement si les données SeizeIT2 sont déjà disponibles dans `data/`.

Le repo ne contient pas les fichiers EDF bruts. Placez-les dans Google Drive ou téléchargez-les depuis OpenNeuro, puis montez/copiez les données dans `data/`.

In [ ]:
# Décommentez si vous voulez monter Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

# Exemple : copier un dossier préparé depuis Drive vers le repo.
# !mkdir -p data
# !rsync -av /content/drive/MyDrive/SeizeIT2_subset/ data/

In [ ]:
# Installation lourde pour la reproduction complète.
# À lancer seulement après ajout des fichiers de données.

# !pip install -q -r requirements-pipeline.txt
# !python src/load_data.py
# !python src/preprocess.py
# !python src/pipeline_multirun.py
# !python src/train_multirun.py
# !python src/train_mlp.py
# !python src/estimate_esp32_cost.py
# !python src/make_figures.py

## 8. Mettre à jour le site après modification du repo

Après un `git pull` ou après avoir copié une nouvelle présentation dans `assets/presentation_iot.pdf`, relancez simplement la cellule de lancement Streamlit.

In [ ]:
# Exemple de mise à jour rapide
# !git pull
# !pkill -f "streamlit run" || true
# !nohup streamlit run streamlit_app.py --server.port 8501 --server.address 0.0.0.0 > /tmp/streamlit.log 2>&1 &
# !sleep 4
# !tail -n 30 /tmp/streamlit.log